## Data

In [ ]:
!pwd

In [ ]:
# https://www.dol.gov/agencies/eta/foreign-labor/performance

In [ ]:
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2021_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2021_Q1.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2021_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2021_Q2.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2021_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2021_Q3.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2021_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2021_Q4.xlsx

In [ ]:
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2022_Q1.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2022_Q2.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2022_Q3.xlsx
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2022_Q4.xlsx

In [ ]:
! wget -P /opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2023_Q1.xlsx

## Analysis

### Spark

In [ ]:
import findspark 
findspark.init() 

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StringType, StructField, TimestampType, DoubleType, ArrayType, StringType, MapType, BinaryType
from delta import *

import json
import os
import getpass

import numpy as np
import pandas as pd

In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("H1B_Sal_Analysis").getOrCreate()
# .config("spark.jars.packages", "com.crealytics:spark-excel_2.12:3.3.1_0.18.5")

sc = spark.sparkContext
sc.setLogLevel('ERROR')

In [ ]:
# maxRowsInMemory = 20,  // Optional, default None. If set, uses a streaming reader which can help with big files (will fail if used with xls format files)
# maxByteArraySize = 2147483647,  // Optional, default None. See https://poi.apache.org/apidocs/5.0/org/apache/poi/util/IOUtils.html#setByteArrayMaxOverride-int-
# tempFileThreshold = 10000000, // Optional, default None. Number of bytes at which a zip entry is regarded as too large for holding in memory and the data is put in a temp file instead

In [ ]:
df = spark.read.format("com.crealytics.spark.excel") \
.option("header", "true") \
.option("inferSchema", "true") \
.option("maxRowsInMemory", "20") \
.option("maxByteArraySize", "2147483647") \
.option("tempFileThreshold", "100000000") \
.load("/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2022_Q1.xlsx")

In [ ]:
df.limit(1).toPandas()

In [ ]:
spark.sql("show databases;").show()

In [ ]:
spark.sql("create database if not exists test_db;")

In [ ]:
spark.sql("drop table if exists test_db.lca_2022")

In [ ]:
# for i in range(1, 5):
#     f = f"/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2022_Q{i}.xlsx"
#     print(f)
#     spark.read.format("com.crealytics.spark.excel") \
#     .option("header", "true") \
#     .option("inferSchema", "true") \
#     .option("maxRowsInMemory", "20") \
#     .option("maxByteArraySize", "2147483647") \
#     .option("tempFileThreshold", "100000000") \
#     .load(f) \
#     .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("test_db.lca_2022")
#     print(spark.sql("select count(1) from test_db.lca_2022").toPandas())

In [ ]:
spark.sql("show tables in test_db").toPandas()

In [ ]:
spark.sql("drop table if exists test_db.lca_2022_csv")

spark.read.csv([f'/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2022_Q{i}.csv' for i in range(1, 5)], header = True) \
.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("test_db.lca_2022_csv")

In [ ]:
spark.read.csv('/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2023_Q1.csv', header = True) \
.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("test_db.lca_2022_csv")

In [ ]:
spark.sql("select count(1) from test_db.lca_2022_csv").toPandas()

In [ ]:
spark.sql("select min(received_date), max(received_date) from test_db.lca_2022_csv").toPandas()

In [ ]:
spark.sql("""
select job_title, worksite_city, worksite_address1, worksite_county, worksite_state
, WAGE_RATE_OF_PAY_FROM, WAGE_RATE_OF_PAY_TO, PREVAILING_WAGE, BEGIN_DATE, *
from test_db.lca_2022_csv 
where 1=1
and lower(employer_name) like '%shipt%' 
--and upper(WORKSITE_STATE) = 'KS' 
and lower(JOB_TITLE) like '%data%engineer%'
and lower(JOB_TITLE) not like '%manager%'
and lower(JOB_TITLE) not like '%director%'
order by begin_date desc
limit 100
""").toPandas()

In [ ]:
spark.sql("select cast(received_date as date) as received_date, count(1) as num_apps from test_db.lca_2022_csv group by 1").toPandas().plot.line(x = 'received_date', figsize=(12, 6))

In [ ]:
spark.sql("select cast(received_date as date) as received_date, count(1) as num_apps from test_db.lca_2022_csv where received_date > '2021-08-31' group by 1").toPandas().plot.line(x = 'received_date', figsize=(12, 6))

In [ ]:
spark.sql("select employer_name from test_db.lca_2022_csv where received_date > '2021-08-31' group by 1").count()

In [ ]:
spark.sql("""
select job_title 
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%shipt%' 
and lower(employer_name) not like '%shiptracks%'
group by 1
""").toPandas()

In [ ]:
spark.sql("""
select * 
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%shipt%' 
and lower(employer_name) not like '%shiptracks%'
and lower(job_title) like '%machine%'

""").toPandas()

In [ ]:
spark.sql("""
select lower(job_title) as job_title
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(job_title) like '%lead%data%engineer%'
and lower(job_title) not like '%director%'
and lower(job_title) not like '%test%'
group by 1
""") \
.toPandas()

In [ ]:
spark.sql("""
select cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(job_title) like '%lead%data%engineer%'
and lower(job_title) not like '%director%'
and lower(job_title) not like '%test%'
--group by 1
""") \
.toPandas() \
.plot.hist(bins = 30, figsize=(12, 6))

In [ ]:
spark.sql("""
select cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(job_title) like '%lead%data%engineer%'
and lower(job_title) not like '%director%'
and lower(job_title) not like '%test%'
--group by 1
""") \
.toPandas() \
.describe()

In [ ]:
spark.sql("""
select cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary
from test_db.lca_2022_csv 
where 1=1
--and received_date > '2021-08-31' 
and lower(job_title) like '%staff%data%engineer%'
and lower(job_title) not like '%director%'
and lower(job_title) not like '%test%'
--group by 1
""") \
.toPandas() \
.plot.hist(bins = 24, figsize=(12, 6))

In [ ]:
spark.sql("""
select cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(job_title) like '%staff%data%engineer%'
and lower(job_title) not like '%director%'
and lower(job_title) not like '%test%'
--group by 1
""") \
.toPandas() \
.describe()

In [ ]:
# Instacart

spark.sql("""
select job_title, cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary, *
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%maplebear%' 
and lower(job_title) like '%data%engineer%'
""").toPandas()

In [ ]:
# Uber

spark.sql("""
select job_title, cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary, *
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%uber%' 
and lower(job_title) like '%data%engineer%'
""").toPandas()

In [ ]:
# Doordash

spark.sql("""
select job_title, cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary, *
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%doordash%' 
and lower(job_title) like '%data%engineer%'
""").toPandas()

In [ ]:
# Grubhub

spark.sql("""
select job_title, cast(coalesce(wage_rate_of_pay_to, wage_rate_of_pay_from) as float) as salary, *
from test_db.lca_2022_csv 
where 1=1
and received_date > '2021-08-31' 
and lower(employer_name) like '%grubhub%' 
--and lower(job_title) like '%data%engineer%'
and lower(job_title) like '%manager%'
""").toPandas()

### Pandas

In [ ]:
! pip install openpyxl

In [ ]:
for i in range(1, 5):
    f = f"/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2022_Q{i}"
    print(f)
    pd.read_excel(f + '.xlsx').to_csv(f + '.csv', index = None, header = True)

In [ ]:
f = f"/opt/spark/work-dir/manoj_repo/shipt/storage/spark/2022_salaries/LCA_Disclosure_Data_FY2023_Q1"
print(f)
pd.read_excel(f + '.xlsx').to_csv(f + '.csv', index = None, header = True)

In [ ]:
df = pd.read_excel('lca_Q1_2023.xlsx')

In [ ]:
df.head()

In [ ]:
#for i in df.columns: print(i)

In [ ]:
df['EMPLOYER_NAME'].unique()

In [ ]:
df[df['EMPLOYER_NAME'].apply(lambda x: x.lower()).str.contains('shipt') \
   & df['JOB_TITLE'].str.contains('Data Engineer')] \
.head()[['WORKSITE_ADDRESS1', 'WORKSITE_ADDRESS2', 'WORKSITE_CITY', 'WORKSITE_STATE', 'WAGE_RATE_OF_PAY_FROM', 'PREVAILING_WAGE']]

In [ ]:
df[df['EMPLOYER_NAME'].apply(lambda x: x.lower()).str.contains('shipt') \
   & df['JOB_TITLE'].apply(lambda x: x.lower()).str.contains('data engineer')]

#### Lead Data Engineer

In [ ]:
lead_data_engineer = df[df['JOB_TITLE'].apply(lambda x: x.lower()).str.contains('lead data engineer')] \
[['EMPLOYER_NAME', 
  'WORKSITE_CITY', 
  'WORKSITE_STATE', 
  'WAGE_RATE_OF_PAY_FROM', 
  'WAGE_RATE_OF_PAY_TO',   
  'PREVAILING_WAGE']]

lead_data_engineer.head(2)

In [ ]:
lead_data_engineer.shape

In [ ]:
lead_data_engineer['WAGE_RATE_OF_PAY_TO'] = lead_data_engineer['WAGE_RATE_OF_PAY_TO'].fillna(lead_data_engineer['WAGE_RATE_OF_PAY_FROM'])

In [ ]:
lead_data_engineer['WAGE_RATE_OF_PAY_TO'].describe()

In [ ]:
lead_data_engineer['WAGE_RATE_OF_PAY_TO'].plot.hist()

In [ ]:
lead_data_engineer[lead_data_engineer['EMPLOYER_NAME'].apply(lambda x: x.lower()).str.contains('maplebear')]

#### Staff Data Engineer

In [ ]:
staff_data_engineer = df[df['JOB_TITLE'].apply(lambda x: x.lower()).str.contains('staff data engineer')] \
[['EMPLOYER_NAME', 
  'WORKSITE_CITY', 
  'WORKSITE_STATE', 
  'WAGE_RATE_OF_PAY_FROM', 
  'WAGE_RATE_OF_PAY_TO',   
  'PREVAILING_WAGE']]

staff_data_engineer.head(2)

In [ ]:
staff_data_engineer.shape

In [ ]:
staff_data_engineer['WAGE_RATE_OF_PAY_TO'] = staff_data_engineer['WAGE_RATE_OF_PAY_TO'].fillna(staff_data_engineer['WAGE_RATE_OF_PAY_FROM'])

In [ ]:
staff_data_engineer['WAGE_RATE_OF_PAY_TO'].describe()

In [ ]:
staff_data_engineer['WAGE_RATE_OF_PAY_TO'].plot.hist()